## 01 — QLoRA Fine-Tuning: Mistral-7B-Instruct-v0.2 on CaseHOLD

Fine-tunes `mistralai/Mistral-7B-Instruct-v0.2` with QLoRA (4-bit NF4 quantization, LoRA rank 64, alpha 16, dropout 0.03) on the LexGLUE CaseHOLD dataset. Training ran for 2 epochs on a 400-example subset of the 45,000-example train split on Google Colab (T4 GPU). Training loss curves and generated predictions are preserved in the cell outputs below.

**Note on inference metrics:** The inference loop was interrupted by the Colab session. Evaluation metrics (ROUGE-L and cosine similarity) in the cells below are computed on a partial sample of approximately 221–239 of the 3,600 test examples — not the full test set.

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

In [2]:
from huggingface_hub import notebook_login # hf_REDACTED
notebook_login()

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="bfloat16"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [4]:
model.gradient_checkpointing_enable()
model.config.use_cache = False  # required for checkpointing

In [20]:
# --- 2. Add LoRA adapters ---
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=64,#64
    lora_alpha=16,
    lora_dropout=0.03#0.05
)
model.enable_input_require_grads()
model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [21]:
dataset = load_dataset("lex_glue", "case_hold")

In [22]:
tokenizer.pad_token = tokenizer.eos_token

In [23]:
dataset["train"]["label"]

Column([0, 1, 4, 0, 3])

In [24]:
# --- 4. Tokenization ---
def tokenize_casehold(example):
    """
    Input: legal context
    Output: case holding
    """
    input_text = f"Context: {example['context']}\nQuestion: What is the holding?\nAnswer:"
    output_text = example['endings'][example['label']]

    # Tokenize inputs
    input_encodings = tokenizer(
        input_text,
        truncation=True,
        padding="max_length",
        max_length=256--128
    )

    # Tokenize labels with same length
    label_encodings = tokenizer(
        output_text,
        truncation=True,
        padding="max_length",
        max_length=256--128
    )


    # Replace pad tokens in labels with -100
    labels = [
        (token_id if token_id != tokenizer.pad_token_id else -100)
        for token_id in label_encodings["input_ids"]
    ]

    return {
    "input_ids": input_encodings["input_ids"],
    "attention_mask": input_encodings["attention_mask"],
    "labels": labels  # <--- labels come from label_encodings
}
tokenized_dataset = dataset.map(
    tokenize_casehold,
    remove_columns=dataset["train"].column_names
)


#tokenized_dataset = dataset.map(tokenize_casehold, remove_columns=dataset["train"].column_names)


Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3600 [00:00<?, ? examples/s]

Map:   0%|          | 0/3900 [00:00<?, ? examples/s]

In [25]:
len(tokenized_dataset["train"]["labels"][0])

384

In [45]:
# --- 5. Training arguments ---
training_args = TrainingArguments(
    output_dir="./qlora-lexglue-casehold",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,  # increase for full training
    logging_steps=10,
    save_strategy="epoch",
    bf16=torch.cuda.is_available(),
    optim="paged_adamw_32bit",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none" ,  # ✅ disable wandb

)

In [46]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
#from transformers import DataCollatorForSeq2Seq

#data_collator = DataCollatorForSeq2Seq(
 #   tokenizer=tokenizer,
  #  model=model,
   # label_pad_token_id=-100
#)
# --- 6. Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(400)),  # small subset for Colab

    eval_dataset=tokenized_dataset["validation"].select(range(100)),
    data_collator=data_collator

)

In [47]:
#memory management
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [48]:
# --- 7. Train ---


model.config.use_cache = False
trainer.train()
model.config.use_cache = True

Epoch,Training Loss,Validation Loss
1,1.611100,1.624055


Epoch,Training Loss,Validation Loss
1,1.611100,1.624055
2,1.460700,1.637933


In [49]:
# inference_pipeline2_run.py
import torch
import json
from transformers import AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

test_dataset = dataset["test"]


In [50]:
# Enable eval mode
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): MistralForCausalLM(
          (model): MistralModel(
            (embed_tokens): Embedding(32000, 4096)
            (layers): ModuleList(
              (0-31): 32 x MistralDecoderLayer(
                (self_attn): MistralAttention(
                  (q_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.03, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=4096, out_features=64, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=64, out_features=4096, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
    

In [51]:

# --- 3. Run inference ---
results = []
max_new_tokens = 64  # adjust if needed

for i, example in enumerate(test_dataset):
    gold = example["endings"][example["label"]]  # correct holding

    # Build input prompt
    input_text = f"Context: {example['context']}\nQuestion: What is the holding?\nAnswer:"

    # Tokenize
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    # Generate prediction
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Decode prediction
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = pred.replace(input_text, "").strip()  # remove prompt

    # Save result
    results.append({
        "id": str(i),
        "context": example["context"],
        "gold_holding": gold,
        "predicted_answer": pred
    })

    if i % 20 == 0:
        print(f"Processed {i} examples...")


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Processed 0 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

Processed 20 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

...

Processed 200 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

Processed 220 examples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

KeyboardInterrupt: 

**Note on results:** The inference loop was interrupted before completion. Metrics below are computed on a partial sample of approximately 221–239 of 3,600 test examples. Results should be read as preliminary, not full-dataset evaluation.

In [52]:
from datasets import load_dataset
import pandas as pd
import evaluate
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- 5. Compute metrics (ROUGE-L + Cosine Similarity) ---
#rouge = load_metric("rouge")
rouge = evaluate.load("rouge")
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")#("all-MiniLM-L6-v2")

rouge_scores, cosine_scores = [], []

for row in results:
    pred = row["predicted_answer"]
    gold = row["gold_holding"]

    # ROUGE-L
    #rouge_l = rouge.compute(predictions=[pred], references=[gold])["rougeL"].mid.fmeasure
    rouge_result = rouge.compute(predictions=[pred], references=[gold], rouge_types=["rougeL"])
    rouge_l = rouge_result["rougeL"]  # this will be a float

    rouge_scores.append(rouge_l)

    # Cosine similarity
    emb = embedder.encode([pred, gold])
    cos_sim = cosine_similarity([emb[0]], [emb[1]])[0][0]
    cosine_scores.append(cos_sim)

# Save metrics CSV
df_metrics = pd.DataFrame({
    "id": [row["id"] for row in results],
    "gold": [row["gold_holding"] for row in results],
    "predicted": [row["predicted_answer"] for row in results],
    "rougeL": rouge_scores,
    "cosine": cosine_scores
})
##df_metrics.to_csv("pipeline2_eval.csv", index=False)
print(f"✅ Saved evaluation metrics CSV (ROUGE-L + Cosine) to pipeline2_eval.csv")
print("Avg ROUGE-L:", sum(rouge_scores)/len(rouge_scores))
print("Avg Cosine:", sum(cosine_scores)/len(cosine_scores))



✅ Saved evaluation metrics CSV (ROUGE-L + Cosine) to pipeline2_eval.csv
Avg ROUGE-L: 0.211253807828264
Avg Cosine: 0.565993


In [52]:
# --- 6. Prepare human evaluation CSV (sample 50) ---
import random
sample_size = 10
sampled = random.sample(results, sample_size)

df_human = pd.DataFrame([{
    "id": row["id"],
    "context": row["context"][:500] + "...",
    "gold_holding": row["gold_holding"],
    "predicted_answer": row["predicted_answer"]
} for row in sampled])

#df_human.to_csv("pipeline2_human_eval.csv", index=False)
print(f"✅ Saved human evaluation CSV (sample {sample_size}) to pipeline2_human_eval.csv")

✅ Saved human evaluation CSV (sample 10) to pipeline2_human_eval.csv


In [53]:
from datasets import load_dataset
import pandas as pd
import evaluate
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------
# 1. Load dataset results
# -------------------------
# assumes you already have "results" from inference
# results = [...]

# -------------------------
# 2. Load ROUGE metric
# -------------------------
rouge = evaluate.load("rouge")

# -------------------------
# 3. Load Legal-BERT for embeddings
# -------------------------
tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
embedder_model = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased")

def get_legalbert_embedding(text: str):
    """Compute Legal-BERT sentence embedding via mean pooling."""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = embedder_model(**inputs)
    # Mean pooling: average across token embeddings
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding

# -------------------------
# 4. Compute metrics
# -------------------------
rouge_scores, cosine_scores = [], []

for row in results:
    pred = row["predicted_answer"]
    gold = row["gold_holding"]

    # ROUGE-L
    rouge_result = rouge.compute(predictions=[pred], references=[gold], rouge_types=["rougeL"])
    rouge_l = rouge_result["rougeL"]
    rouge_scores.append(rouge_l)

    # Cosine similarity with Legal-BERT embeddings
    emb_pred = get_legalbert_embedding(pred)
    emb_gold = get_legalbert_embedding(gold)
    cos_sim = cosine_similarity([emb_pred], [emb_gold])[0][0]
    cosine_scores.append(cos_sim)

# -------------------------
# 5. Save metrics
# -------------------------
df_metrics = pd.DataFrame({
    "id": [row["id"] for row in results],
    "gold": [row["gold_holding"] for row in results],
    "predicted": [row["predicted_answer"] for row in results],
    "rougeL": rouge_scores,
    "cosine": cosine_scores
})
#df_metrics.to_csv("pipeline2_eval_legalbert.csv", index=False)

print(f"✅ Saved evaluation metrics CSV (ROUGE-L + Cosine) to pipeline2_eval_legalbert.csv")
print("Avg ROUGE-L:", sum(rouge_scores)/len(rouge_scores))
print("Avg Cosine:", sum(cosine_scores)/len(cosine_scores))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

✅ Saved evaluation metrics CSV (ROUGE-L + Cosine) to pipeline2_eval_legalbert.csv
Avg ROUGE-L: 0.211253807828264
Avg Cosine: 0.8961609
